# HyperView: CLIP vs HyCoCLIP with 2D UMAP Projection on CIFAR-100

This notebook compares two embedding spaces on the same image set.

- [**CLIP**](https://github.com/openai/CLIP) maps images and text into one shared vector space.
    
- [**HyCoCLIP**](https://github.com/PalAvik/hycoclip) maps images into a space designed for hierarchy and tree like structure.
    
- We then compute a **2D projection** of each embedding space with [UMAP](https://umap-learn.readthedocs.io/en/latest/) so we can inspect clusters and label structure.
    

HyperView allows us to compare the embedding spaces of these different geometries.

In this demo, we use the [CIFAR-100](https://huggingface.co/datasets/uoft-cs/cifar100) dataset of tiny images, available through HuggingFace.

## Install

HyperView is a library for dataset curation and model analysis. It provides tools for interactive visualization of the embedding space.

In [1]:
%%capture
# If you run this in Google Colab, install HyperView with this command.
!uv pip install hyperview

## Import HyperView


We import HyperView and use it as the main interface for:

- reading data
    
- computing embeddings with pretrained models
    
- computing a 2D projection for visualization
    
- launching the interactive viewer
    

In [2]:
import hyperview as hv

## Configuration

We specify the dataset and model settings in a single dictionary.

### Data source

- We load **CIFAR-100** from Hugging Face.
    
- We use the **test** split.
    
- `img` is the image field.
    
- We use `coarse_label` for labels.

Why coarse labels:

- HyperView disables distinct label coloring when there are more than 20 labels.
    
- CIFAR-100 has 100 fine labels, but 20 coarse labels.
    
- Using coarse labels keeps label coloring useful in the viewer.
    

### Sampling

- `NUM_SAMPLES = 200` keeps embedding and layout computation fast enough for a demo.
    
- Increase this if you want denser clusters, but expect more compute time.
    

### Models

- `openai/clip-vit-base-patch32` is a common CLIP baseline.
    
- `hycoclip-vit-s` is a HyCoCLIP model that targets hierarchical structure.

In [ ]:
DATASET_NAME = "cifar100_coarse_clip_hyper_models"
HF_DATASET = "uoft-cs/cifar100"
HF_SPLIT = "test"
HF_IMAGE_KEY = "img"
# NOTE: HyperView disables distinct label coloring when there are >20 labels.
# CIFAR-100 has 100 fine labels, but only 20 coarse labels.
HF_LABEL_KEY = "coarse_label"
NUM_SAMPLES = 200
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
HYPER_MODELS_MODEL_ID = "hycoclip-vit-s"

## Load CIFAR-100 into a HyperView dataset

We create a HyperView `Dataset` object and then import samples from Hugging Face.

Key parameters:

- `persist=False` keeps this dataset in memory for this run.
    
- `max_samples=NUM_SAMPLES` subsamples the split.
    

At the end, `len(dataset)` is the number of loaded examples.

In [3]:
print("Loading CIFAR-100 from Hugging Face...")
dataset = hv.Dataset(DATASET_NAME, persist=False)
dataset.add_from_huggingface(
    HF_DATASET,
    split=HF_SPLIT,
    image_key=HF_IMAGE_KEY,
    label_key=HF_LABEL_KEY,
    max_samples=NUM_SAMPLES,
)
print(f"Loaded {len(dataset)} samples")



Loading CIFAR-100 from Hugging Face...


README.md: 0.00B [00:00, ?B/s]

cifar100/train-00000-of-00001.parquet:   0%|          | 0.00/119M [00:00<?, ?B/s]

cifar100/test-00000-of-00001.parquet:   0%|          | 0.00/23.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loading 200 samples from uoft-cs/cifar100...
Images saved to: /root/.hyperview/media/huggingface/uoft-cs_cifar100/test
Loaded 200 samples
Computing embeddings for 200 samples...
Computing euclidean umap layout for 200 samples...
Computing embeddings for 200 samples...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Computing poincare umap layout for 200 samples...
Launching at http://127.0.0.1:6262

HyperView is running (Colab, port=6262). Use the link below to open it.


## Compute CLIP embeddings

### What CLIP embeddings represent

CLIP trains an image encoder and a text encoder so that matching image text pairs have nearby vectors. When we compute image embeddings here, each image becomes a single vector in a shared space that also supports text vectors.

### What we do in code

- `compute_embeddings(CLIP_MODEL_ID)` runs the CLIP image encoder and stores one vector per image.
    
- The returned `space_key` is a handle to that embedding space inside HyperView.
    

### Why we compute a 2D visualization

High dimensional vectors are hard to inspect. We compute a 2D layout so we can see neighborhood structure.

- `compute_visualization(..., geometry="euclidean")` treats the embedding space as Euclidean.
    
- This matches how CLIP vectors are often used with cosine similarity or dot product in a flat vector space.
    

Note on “UMAP”  
This notebook focuses on a 2D projection step. HyperView computes a 2D layout for the viewer. The title calls this UMAP. In practice, you should treat this as a nonlinear projection that preserves local neighborhoods better than PCA.

## Compute HyCoCLIP embeddings

### Why a different geometry

Some datasets have hierarchical label structure. A tree structure is hard to represent in Euclidean space without distortion.

Hyperbolic spaces can represent tree growth patterns with less distortion than Euclidean spaces. A common model is the **Poincaré ball**, which is a way to work with hyperbolic geometry in a bounded region.

### What we do in code

- `compute_embeddings(model=HYPER_MODELS_MODEL_ID)` computes embeddings from the HyCoCLIP model.
    
- `compute_visualization(..., geometry="poincare")` tells HyperView to treat distances and neighborhoods using Poincaré geometry.
    

How to read the plot

- In Poincaré style layouts, points near the center and points near the boundary can have different distance behavior than Euclidean layouts.
    
- Focus on neighborhood membership and cluster separation rather than raw coordinate scale.

In [ ]:
clip_space = dataset.compute_embeddings(CLIP_MODEL_ID)
dataset.compute_visualization(space_key=clip_space, geometry="euclidean")
hyper_space = dataset.compute_embeddings(model=HYPER_MODELS_MODEL_ID)
dataset.compute_visualization(space_key=hyper_space, geometry="poincare")

## Launch the interactive viewer

`hv.launch(dataset, open_browser=True)` starts a local server and opens the viewer.

In the UI, you should be able to:

- switch between embedding spaces (CLIP vs HyCoCLIP)
    
- inspect image thumbnails for selected points
    
- compare how coarse labels cluster under each geometry
    


In [ ]:
hv.launch(dataset, open_browser=True)

## Suggested checks

- Are nearest neighbors under Euclidean CLIP similar to nearest neighbors under Poincaré HyCoCLIP?


    